In [0]:
dbutils.widgets.text("year", "2026")
dbutils.widgets.text("month", "01")
dbutils.widgets.text("file_name", "2026-1.csv")
dbutils.widgets.text("run_id", "manual-dev")

year = dbutils.widgets.get("year")
month = dbutils.widgets.get("month")
file_name = dbutils.widgets.get("file_name")
run_id = dbutils.widgets.get("run_id")

source_path = (
    f"abfss://landing@stchilecompradev.dfs.core.windows.net/"
    f"chilecompra/ordenes_compra/historical/"
    f"year={year}/month={month}/{file_name}"
)


In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load(source_path)
)


In [0]:
import re
import unicodedata


def normalize_column_name(name):
    # Quitar acentos
    name = unicodedata.normalize("NFKD", name)
    name = "".join(
        char for char in name
        if not unicodedata.combining(char)
    )

    # minúsculas
    name = name.lower().strip()

    # reemplazar caracteres especiales por _
    name = re.sub(r"[^a-z0-9_]+", "_", name)

    # evitar varios ___ seguidos
    name = re.sub(r"_+", "_", name)

    return name.strip("_")

In [0]:
df = df.toDF(
    *[normalize_column_name(column) for column in df.columns]
)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_bronze = (
    df
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit(file_name))
    .withColumn("_source_year", lit(year).cast("int"))
    .withColumn("_source_month", lit(month).cast("int"))
    .withColumn("_pipeline_run_id", lit(run_id))
)


In [0]:
target_table = "chilecompra.bronze.ordenes_compra_historical"

if spark.catalog.tableExists(target_table):
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"_source_year = {int(year)} AND _source_month = {int(month)}"
        )
        .saveAsTable(target_table)
    )
else:
    (
        df_bronze.write
        .format("delta")
        .partitionBy("_source_year", "_source_month")
        .saveAsTable(target_table)
    )

In [0]:
from pyspark.sql.functions import col

source_count = df_bronze.count()

written_count = (
    spark.table(target_table)
    .filter(
        (col("_source_year") == int(year)) &
        (col("_source_month") == int(month))
    )
    .count()
)

if written_count != source_count:
    raise ValueError(
        f"DQ FAILED: expected {source_count} rows "
        f"but Bronze contains {written_count}"
    )

print(
    f"Bronze load completed: "
    f"year={year}, month={month}, rows={written_count}"
)